In [109]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import DataConversionWarning
from sklearn.preprocessing import RobustScaler
from dowhy import CausalModel
from IPython.display import display


warnings.filterwarnings("ignore", category=DataConversionWarning)

In [110]:
df = pd.read_csv("alerts_with_category_with_apk_size_updated.csv")
print("Original rows:", len(df))

df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
print("After removing obfuscated:", len(df))

df["verdict"] = df["verdict"].astype(int)

download_order = ['<100','100-500','500-1k','1k-5k','5k-10k','10k-50k',
                  '50k-100k','100k-500k','500k-1M','1M-5M','>5M']
ord_map = {c:i for i,c in enumerate(download_order)}
df["app_popularity_encoded"] = df["app_popularity"].map(ord_map)
df = df.dropna(subset=["app_popularity_encoded"]).copy()
df["app_popularity_encoded"] = df["app_popularity_encoded"].astype(int)

scaler = RobustScaler()
df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])


Original rows: 6140
After removing obfuscated: 5612


In [111]:
small_libs = ["SocialMedia", "Analytics", "Cloud"]
df["lib_grouped"] = df["code_location"].replace(small_libs, "Small_Libraries")
df.loc[df["lib_grouped"] == "developer_written", "lib_grouped"] = "App_Source_Code"

print("\nGrouped library counts:")
print(df["lib_grouped"].value_counts())

baseline_name = "App_Source_Code"
libs = [baseline_name] + [l for l in df["lib_grouped"].unique() if l != baseline_name]
print("\nLibrary levels:")
for l in libs:
    print("-", l)


Grouped library counts:
lib_grouped
Utilities          3039
others             1341
App_Source_Code     511
Android             494
Small_Libraries     227
Name: count, dtype: int64

Library levels:
- App_Source_Code
- others
- Android
- Utilities
- Small_Libraries


In [112]:
refutation_methods = [
    "random_common_cause",
    "placebo_treatment_refuter",
    "data_subset_refuter",
]

In [113]:
def make_pairwise_balanced(df_in, lib_name, baseline="App_Source_Code", random_state=42):
    df_sub = df_in[df_in["lib_grouped"].isin([baseline, lib_name])].copy()

    base = df_sub[df_sub["lib_grouped"] == baseline]
    lib  = df_sub[df_sub["lib_grouped"] == lib_name]

    n_min = min(len(base), len(lib))

    base_bal = base.sample(n=n_min, replace=False, random_state=random_state)
    lib_bal  = lib.sample(n=n_min, replace=False, random_state=random_state)

    df_bal = pd.concat([base_bal, lib_bal], axis=0)
    df_bal = df_bal.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df_bal


In [114]:
def apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate):
    ref_rows = []
    for method in refutation_methods:
        if method == "placebo_treatment_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, placebo_type="permute", num_simulations=20)
        elif method == "data_subset_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, subset_fraction=0.8, num_simulations=20)
        else:
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method)

        print(f"\nRefuter: {method}\n", ref)

        ref_rows.append({
            "library": lib_name,
            "setting": label,
            "refuter": method,
            "orig_effect": ate,
            "new_effect": getattr(ref, "new_effect", None),
            "p_value": getattr(ref, "p_value", None)
        })
        
    return ref_rows

In [115]:
def run_library_contrast(df_sub, lib_name, baseline="App_Source_Code", label="unbalanced"):
    df_sub = df_sub.copy()
    df_sub["treatment_lib"] = (df_sub["lib_grouped"] == lib_name).astype(int)

    n0 = int((df_sub["treatment_lib"] == 0).sum())
    n1 = int((df_sub["treatment_lib"] == 1).sum())

    print("\n" + "-"*14 + f" {lib_name} vs {baseline} ({label}) " + "-"*14)
    print("Counts:", {baseline: n0, lib_name: n1})


    causal_graph = """
    digraph {
        treatment_lib -> verdict;
        app_popularity_encoded -> treatment_lib;
        app_popularity_encoded -> verdict;
        apk_size_scaled -> treatment_lib;
       app_popularity_encoded -> apk_size_scaled;
    }
    """

    model = CausalModel(
        data=df_sub,
        treatment="treatment_lib",
        outcome="verdict",
        graph=causal_graph.replace("\n", " "),
        common_causes=["app_popularity_encoded"]
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.propensity_score_matching",
        target_units="ate",
        confidence_intervals='bootstrap',
        method_params={
            "num_simulations": 300,
            "sample_size_fraction": 1.0,
            "confidence_level": 0.95,
        },
    )

    ate = float(estimate.value)
    try:
        ci_low, ci_high = estimate.get_confidence_intervals()
        ci_low, ci_high = float(ci_low), float(ci_high)
    except Exception:
        ci_low, ci_high = None, None

    print("ATE:", ate)
    if ci_low is not None:
        print("95% CI:", (ci_low, ci_high))


    ref_rows = apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate)

    result_row = {
        "library": lib_name,
        "setting": label,
        "n_baseline": n0,
        "n_lib": n1,
        "ATE": ate,
        "CI_low": ci_low,
        "CI_high": ci_high
    }

    return result_row, ref_rows




In [116]:
main_rows = []
ref_rows_all = []

for lib in libs:
    if lib == baseline_name:
        continue

    # Unbalanced subset
    df_unbal = df[df["lib_grouped"].isin([baseline_name, lib])].copy()
    main_row_u, ref_u = run_library_contrast(df_unbal, lib, baseline=baseline_name, label="unbalanced")
    main_rows.append(main_row_u)
    ref_rows_all.extend(ref_u)

    # # Balanced subset
    # df_bal = make_pairwise_balanced(df, lib, baseline=baseline_name, random_state=42)
    # main_row_b, ref_b = run_library_contrast(df_bal, lib, baseline=baseline_name, label="balanced")
    # main_rows.append(main_row_b)
    # ref_rows_all.extend(ref_b)



-------------- others vs App_Source_Code (unbalanced) --------------
Counts: {'App_Source_Code': 511, 'others': 1341}
ATE: -0.05453563714902808
95% CI: (-0.22570194384449244, 0.20734341252699778)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:-0.05453563714902808
New effect:-0.054535637149028086
p value:1.0


Refuter: placebo_treatment_refuter
 Refute: Use a Placebo Treatment
Estimated effect:-0.05453563714902808
New effect:0.013093952483801297
p value:0.44051495948537533


Refuter: data_subset_refuter
 Refute: Use a subset of data
Estimated effect:-0.05453563714902808
New effect:-0.10910931174089074
p value:0.29771323551539297


-------------- Android vs App_Source_Code (unbalanced) --------------
Counts: {'App_Source_Code': 511, 'Android': 494}
ATE: 0.22388059701492538
95% CI: (0.14626865671641792, 0.518407960199005)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:0.22388059701492538
New effect:0.2238805970149254
p

In [117]:
main_df = pd.DataFrame(main_rows).sort_values(["library", "setting"])
ref_df = pd.DataFrame(ref_rows_all)

print("\n\n","-"*25 + "Detailed (ATEs) " + "-"*25)
print(main_df[["library","setting","n_baseline","n_lib","ATE","CI_low","CI_high"]])

pivot = main_df.pivot(index="library", columns="setting", values="ATE")
print("\n\n","-"*25 + "Summary ATE (unbalanced vs balanced) " + "-"*25)
print(pivot)



 -------------------------Detailed (ATEs) -------------------------
           library     setting  n_baseline  n_lib       ATE    CI_low  \
1          Android  unbalanced         511    494  0.223881  0.146269   
3  Small_Libraries  unbalanced         511    227  0.288618  0.173442   
2        Utilities  unbalanced         511   3039 -0.317183 -0.480282   
0           others  unbalanced         511   1341 -0.054536 -0.225702   

    CI_high  
1  0.518408  
3  0.401084  
2 -0.019718  
0  0.207343  


 -------------------------Summary ATE (unbalanced vs balanced) -------------------------
setting          unbalanced
library                    
Android            0.223881
Small_Libraries    0.288618
Utilities         -0.317183
others            -0.054536
